# ProtVS Quickstart

**ProtVS** predicts protein localization images from three reference microscopy
channels (nucleus, ER, microtubules) using a conditional diffusion model.

**Workflow:**
1. Download pre-trained checkpoints (first time only)
2. Initialize the model
3. Explore available proteins and cell lines
4. Load and normalize a reference image
5. Predict protein localization
6. Visualize and save results


## Setup

In [ ]:
import numpy as np
from tifffile import imread

from protvs import ProtVS
from protvs.data import ImageNormalizer


## Step 1: Download Checkpoints

Downloads model weights into the `protvs/` package directory.
**Skip this cell if you have already downloaded them.**


In [ ]:
ProtVS.download_checkpoints()


## Step 2: Initialize Model

Model weights are **lazy-loaded** — they are only moved to GPU on the first
call to `predict()` or `fit()`.


In [ ]:
model = ProtVS()
print(model.summary())


## Step 3: Explore Available Proteins and Cell Lines

Check the model vocabulary before predicting.
Protein and cell line names passed to `predict()` must appear in these lists.


In [ ]:
print(f"Proteins  : {len(model.available_proteins):,}")
print("First 10  :", model.available_proteins[:10])

print(f"\nCell lines: {len(model.available_cell_lines)}")
print(model.available_cell_lines)


## Step 4: Load a Reference Image

The `example_cell_reference_input/` folder in this repo contains pre-normalized
reference images ready for inference — pixel values are already in `[-1, 1]`.

- For **4-channel TIFFs**: channels 0, 2, 3 are used as conditioning
  (microtubules, nucleus, ER); channel 1 is ignored during inference.
- If you want to use **your own raw images**, see `02_preprocessing.ipynb`
  for a full walkthrough of `ChannelAssembler` and `ImageNormalizer`.


In [ ]:
import os

# List available reference images
ref_dir = "example_cell_reference_input"
ref_files = sorted(f for f in os.listdir(ref_dir) if f.endswith(".tif"))
print(f"{len(ref_files)} reference images available:")
for f in ref_files:
    print(" ", f)


In [ ]:
# Load the first reference image (already normalized to [-1, 1])
REF_PATH = os.path.join(ref_dir, ref_files[0])

ref_image = imread(REF_PATH)
print(f"Loaded : {REF_PATH}")
print(f"Shape  : {ref_image.shape}")
print(f"dtype  : {ref_image.dtype}")
print(f"Range  : [{ref_image.min():.2f}, {ref_image.max():.2f}]")


## Step 5: Predict Protein Localization

Pass parallel lists of `(image, protein_name, cell_line_name)` in a single call.
All three lists must have the same length.

- `num_inference_steps`: 50 is the default; use 20 for a quick preview.
- `seed`: set an integer for reproducible results.
- `batch_size`: increase if GPU memory allows; decrease if you run out of memory.


In [ ]:
results = model.predict(
    images=[ref_image, ref_image],
    protein_names=["TOMM20", "BRD4"],
    cell_line_names=["A-431", "A-431"],
    num_inference_steps=50,   # lower (e.g. 20) for a faster preview
    batch_size=4,
    seed=42,                  # remove for non-deterministic sampling
)

print(f"Predictions : {len(results.images)}")
print(f"Output shape: {results.images[0].shape}  "
      f"dtype: {results.images[0].dtype}")


## Step 6: Visualize Results

`show_prediction()` displays all predicted images in a matplotlib figure
with protein name and cell line as titles.


In [ ]:
results.show_prediction()


## Step 7: Save Results

Predictions are saved as **8-bit TIFF** files named:
`{prefix}_{index}_{cell_line}_cell_{protein}.tif`


In [ ]:
results.save_prediction(
    prefix="quickstart",
    directory="predictions/",
)
print("Saved to predictions/")


## Next Steps

**Predict many proteins at once** — pass lists of any length to `model.predict()`;
use `batch_size` to control GPU memory usage:
```python
proteins = ["TOMM20", "BRD4", "LMNA", "ACTB"]
results = model.predict(
    images=[ref_image] * len(proteins),
    protein_names=proteins,
    cell_line_names=["U-2 OS"] * len(proteins),
    batch_size=8,
)
```

**Preprocess your own data** — see `02_preprocessing.ipynb` for a full walkthrough
of `ChannelAssembler` and `ImageNormalizer`.

**Fine-tune on your own data** — provide a directory of 4-channel TIFFs:
```python
model.fit(
    image_dir="data/train",
    image_files=["cell_0.tiff", "cell_1.tiff"],
    protein_names=["CDT1", "TUBB"],
    cell_line_names=["U-2 OS", "U-2 OS"],
    output_dir="finetuned/",
    num_epochs=50,
)
```

**Save and reload a fine-tuned model:**
```python
model.save("finetuned/")
model = ProtVS(checkpoint_dir="finetuned/")
```
